In [1]:
using Symbolics

In [2]:
@variables t (x(t))[1:3] (v(t))[1:3]  # x y z dot(x) dot(y) dot(z)

3-element Vector{Any}:
 t
  (x(t))[1:3]
  (v(t))[1:3]

In [3]:
@variables (y(t))[1:3] (w(t))[1:3]  # r theta phi dot(r) dot(theta) dot(phi)

2-element Vector{Symbolics.Arr{Num, 1}}:
 (y(t))[1:3]
 (w(t))[1:3]

In [4]:
@variables L(x[1],x[2],x[3],v[1],v[2],v[3]) U(x[1],x[2],x[3])

2-element Vector{Num}:
 L((x(t))[1], (x(t))[2], (x(t))[3], (v(t))[1], (v(t))[2], (v(t))[3])
                                  U((x(t))[1], (x(t))[2], (x(t))[3])

In [5]:
@variables m

1-element Vector{Num}:
 m

In [6]:
L = 1//2 * m * (v[1]^2 + v[2]^2 + v[3]^2) - U

-U((x(t))[1], (x(t))[2], (x(t))[3]) + (1//2)*m*((v(t))[1]^2 + (v(t))[2]^2 + (v(t))[3]^2)

In [7]:
@variables X Y Z dx dy dz r θ φ dr dθ dφ

12-element Vector{Num}:
  X
  Y
  Z
 dx
 dy
 dz
  r
  θ
  φ
 dr
 dθ
 dφ

In [8]:
r2y = Dict(
    x[1] => X,
    x[2] => Y,
    x[3] => Z,
    v[1] => dx,
    v[2] => dy,
    v[3] => dz,
    y[1] => r,
    y[2] => θ,
    y[3] => φ,
    w[1] => dr,
    w[2] => dθ,
    w[3] => dφ,
    );

In [9]:
tx = [
    x[1] ~ y[1] * sin(y[2]) * cos(y[3]),
    x[2] ~ y[1] * sin(y[2]) * sin(y[3]),
    x[3] ~ y[1] * cos(y[2])
]

3-element Vector{Equation}:
 (x(t))[1] ~ (y(t))[1]*cos((y(t))[3])*sin((y(t))[2])
 (x(t))[2] ~ (y(t))[1]*sin((y(t))[3])*sin((y(t))[2])
 (x(t))[3] ~ (y(t))[1]*cos((y(t))[2])

In [10]:
for i in 1:3
    display(substitute(tx[i], r2y))
end

X ~ r*cos(φ)*sin(θ)

Y ~ r*sin(φ)*sin(θ)

Z ~ r*cos(θ)

In [11]:
dtdx = Dict()
for i in 1:3
    dtdx[i] = expand_derivatives(Differential(t)(tx[i].rhs))
end

In [25]:
d2w = Dict(
    Differential(t)(y[1]) => w[1],
    Differential(t)(y[2]) => w[2],
    Differential(t)(y[3]) => w[3]
)

v2dtdx = Dict(
    v[1] => substitute(dtdx[1], d2w),
    v[2] => substitute(dtdx[2], d2w),
    v[3] => substitute(dtdx[3], d2w)
)

for i in 1:3
    display(Differential(t)(tx[i].lhs) ~ substitute(dtdx[i], d2w))
end


Differential(t)((x(t))[1]) ~ (w(t))[1]*cos((y(t))[3])*sin((y(t))[2]) + (w(t))[2]*(y(t))[1]*cos((y(t))[3])*cos((y(t))[2]) - (w(t))[3]*(y(t))[1]*sin((y(t))[3])*sin((y(t))[2])

Differential(t)((x(t))[2]) ~ (w(t))[1]*sin((y(t))[3])*sin((y(t))[2]) + (w(t))[2]*(y(t))[1]*cos((y(t))[2])*sin((y(t))[3]) + (w(t))[3]*(y(t))[1]*cos((y(t))[3])*sin((y(t))[2])

Differential(t)((x(t))[3]) ~ (w(t))[1]*cos((y(t))[2]) - (w(t))[2]*(y(t))[1]*sin((y(t))[2])

In [26]:
L2 = simplify(expand(
        substitute(L, v2dtdx)
));

L2 = simplify(L2, rewriter=(
@rule cos(y[2])^2 +
      sin(y[2])^2 * cos(y[3])^2 +
      sin(y[2])^2 * sin(y[3])^2 => 1
))

-U((x(t))[1], (x(t))[2], (x(t))[3]) + (1//2)*m*((w(t))[1]^2)*(cos((y(t))[2])^2) - (1//2)*m*(w(t))[1]*(w(t))[2]*(y(t))[1]*sin(2(y(t))[2]) + (1//2)*m*((w(t))[1]^2)*(cos((y(t))[3])^2)*(sin((y(t))[2])^2) + (1//2)*m*((w(t))[1]^2)*(sin((y(t))[3])^2)*(sin((y(t))[2])^2) + (1//2)*m*(w(t))[1]*(w(t))[2]*(y(t))[1]*(cos((y(t))[3])^2)*sin(2(y(t))[2]) + (1//2)*m*(w(t))[1]*(w(t))[2]*(y(t))[1]*sin(2(y(t))[2])*(sin((y(t))[3])^2) + (1//2)*m*((w(t))[2]^2)*((y(t))[1]^2)*(sin((y(t))[2])^2) + (1//2)*m*((w(t))[2]^2)*((y(t))[1]^2)*(cos((y(t))[3])^2)*(cos((y(t))[2])^2) + (1//2)*m*((w(t))[2]^2)*((y(t))[1]^2)*(cos((y(t))[2])^2)*(sin((y(t))[3])^2) + (1//2)*m*((w(t))[3]^2)*((y(t))[1]^2)*(cos((y(t))[3])^2)*(sin((y(t))[2])^2) + (1//2)*m*((w(t))[3]^2)*((y(t))[1]^2)*(sin((y(t))[3])^2)*(sin((y(t))[2])^2)

In [14]:
substitute(substitute(L2, d2w), r2y)

LoadError: UndefVarError: `d2w` not defined

In [15]:
L3 = 

LoadError: syntax: incomplete: premature end of input

In [32]:
for i in 1:3
    display(
        substitute(
        simplify(expand(expand_derivatives(
                    Differential(t)(Differential(w[i])(L2)) - Differential(y[i])(L2)
        ))),r2y)
            
    )
end

(1//2)*dr*dθ*m*sin(2θ) - dr*m*sin(2θ)*Differential(t)(θ) - (1//2)*dθ*m*sin(2θ)*Differential(t)(r) - (1//2)*m*r*sin(2θ)*Differential(t)(dθ) + m*(cos(θ)^2)*Differential(t)(dr) - dθ*m*r*Differential(t)(θ)*cos(2θ) - (1//2)*dr*dθ*m*sin(2θ)*(sin(φ)^2) - (1//2)*dr*dθ*m*sin(2θ)*(cos(φ)^2) + dr*m*sin(2θ)*(sin(φ)^2)*Differential(t)(θ) + dr*m*sin(2θ)*Differential(t)(θ)*(cos(φ)^2) - (dθ^2)*m*r*(sin(θ)^2) + (1//2)*dθ*m*sin(2θ)*(sin(φ)^2)*Differential(t)(r) + (1//2)*dθ*m*sin(2θ)*Differential(t)(r)*(cos(φ)^2) + (1//2)*m*r*sin(2θ)*(sin(φ)^2)*Differential(t)(dθ) + (1//2)*m*r*sin(2θ)*(cos(φ)^2)*Differential(t)(dθ) + m*(sin(φ)^2)*(sin(θ)^2)*Differential(t)(dr) + m*(cos(φ)^2)*(sin(θ)^2)*Differential(t)(dr) + dθ*m*r*(sin(φ)^2)*Differential(t)(θ)*cos(2θ) + dθ*m*r*Differential(t)(θ)*(cos(φ)^2)*cos(2θ) - (dθ^2)*m*r*(cos(θ)^2)*(sin(φ)^2) - (dθ^2)*m*r*(cos(θ)^2)*(cos(φ)^2) - (dφ^2)*m*r*(sin(φ)^2)*(sin(θ)^2) - (dφ^2)*m*r*(cos(φ)^2)*(sin(θ)^2)

(1//2)*(dr^2)*m*sin(2θ) - (1//2)*dr*m*sin(2θ)*Differential(t)(r) - (1//2)*m*r*sin(2θ)*Differential(t)(dr) + dr*dθ*m*r*cos(2θ) - dr*m*r*Differential(t)(θ)*cos(2θ) - (1//2)*(dr^2)*m*sin(2θ)*(sin(φ)^2) - (1//2)*(dr^2)*m*sin(2θ)*(cos(φ)^2) + (1//2)*dr*m*sin(2θ)*(sin(φ)^2)*Differential(t)(r) + (1//2)*dr*m*sin(2θ)*Differential(t)(r)*(cos(φ)^2) - (1//2)*(dθ^2)*m*(r^2)*sin(2θ) + dθ*m*(r^2)*sin(2θ)*Differential(t)(θ) + (2//1)*dθ*m*r*Differential(t)(r)*(sin(θ)^2) + m*(r^2)*(sin(θ)^2)*Differential(t)(dθ) + (1//2)*m*r*sin(2θ)*(sin(φ)^2)*Differential(t)(dr) + (1//2)*m*r*sin(2θ)*(cos(φ)^2)*Differential(t)(dr) - dr*dθ*m*r*(sin(φ)^2)*cos(2θ) - dr*dθ*m*r*(cos(φ)^2)*cos(2θ) + dr*m*r*(sin(φ)^2)*Differential(t)(θ)*cos(2θ) + dr*m*r*Differential(t)(θ)*(cos(φ)^2)*cos(2θ) + (1//2)*(dθ^2)*m*(r^2)*sin(2θ)*(sin(φ)^2) + (1//2)*(dθ^2)*m*(r^2)*sin(2θ)*(cos(φ)^2) - dθ*m*(r^2)*sin(2θ)*(sin(φ)^2)*Differential(t)(θ) - dθ*m*(r^2)*sin(2θ)*Differential(t)(θ)*(cos(φ)^2) + (2//1)*dθ*m*r*(cos(θ)^2)*(sin(φ)^2)*Differential(t)

dφ*m*(r^2)*sin(2θ)*(sin(φ)^2)*Differential(t)(θ) + dφ*m*(r^2)*sin(2θ)*Differential(t)(θ)*(cos(φ)^2) + 2dφ*m*r*(sin(φ)^2)*Differential(t)(r)*(sin(θ)^2) + 2dφ*m*r*Differential(t)(r)*(cos(φ)^2)*(sin(θ)^2) + m*(r^2)*(sin(φ)^2)*(sin(θ)^2)*Differential(t)(dφ) + m*(r^2)*(cos(φ)^2)*(sin(θ)^2)*Differential(t)(dφ)

In [17]:
substitute(expand_derivatives(Differential(t)(tx[1])), Dict((Differential(t)(y[1])) => w[1]))

0

In [18]:
ty = Dict()
for i in 1:3
    ty[i] = expand_derivatives(sum(w[j] * Differential(y[j])(tx[i]) for j in 1:3))
end    

LoadError: MethodError: no method matching *(::Num, ::SymbolicUtils.BasicSymbolic{Equation})

[0mClosest candidates are:
[0m  *(::Any, ::Any, [91m::Any[39m, [91m::Any...[39m)
[0m[90m   @[39m [90mBase[39m [90m[4moperators.jl:578[24m[39m
[0m  *(::Union{Real, Complex}, [91m::Union{LinearAlgebra.Adjoint{var"#s972", var"#s9721"}, LinearAlgebra.Transpose{var"#s972", var"#s9721"}} where {var"#s972"<:Union{Real, Complex}, var"#s9721"<:(AbstractVector)}[39m, [91m::AbstractMatrix{<:Union{Real, Complex}}[39m, [91m::AbstractMatrix{<:Union{Real, Complex}}[39m)
[0m[90m   @[39m [36mLinearAlgebra[39m [90m/opt/julia-1.9.3/share/julia/stdlib/v1.9/LinearAlgebra/src/[39m[90m[4mmatmul.jl:1218[24m[39m
[0m  *([91m::SpecialFunctions.SimplePoly[39m, ::Any)
[0m[90m   @[39m [32mSpecialFunctions[39m [90m/opt/julia/packages/SpecialFunctions/Zijv9/src/[39m[90m[4mexpint.jl:8[24m[39m
[0m  ...


In [19]:
for i in 1:3
    print("d/dt x^", i, "=")
    display(substitute(ty[i], r2y))
end

d/dt x^1=

LoadError: KeyError: key 1 not found

In [20]:
simplify(expand(substitute(L, Dict(v[1] => ty[1], v[2] => ty[2], v[3] => ty[3]))), rewriter=(@rule cos(y[2])^2 + cos(y[3])^2 * sin(y[2])^2 + sin(y[2])^2 * sin(y[3])^2 => 1))

LoadError: KeyError: key 1 not found

In [21]:
simplify(substitute(simplify(expand(substitute(L, Dict(v[1] => ty[1], v[2] => ty[2], v[3] => ty[3])))), r2y), rewriter=(@rule cos(theta)^2 + cos(phi)^2 * sin(theta)^2 + sin(theta)^2 * sin(phi)^2 => 1))

LoadError: KeyError: key 1 not found

In [22]:
simplify(cos(theta)^2 + cos(phi)^2 * sin(theta)^2 + sin(theta)^2 * sin(phi)^2)

LoadError: UndefVarError: `theta` not defined